In [11]:
#A. Setup and shared data
import sys, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "src" / "model_data.py").exists() else HERE.parent
assert (ROOT / "src" / "model_data.py").exists(), f"Cannot locate repository from {HERE}"
sys.path.insert(0, str(ROOT / "src"))
from model_data import load_features, evaluate, TARGET

OUTPUT = ROOT / "outputs" / "baseline"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Results folder:", OUTPUT)

Repository: C:\Users\gadha\sakura_analytics
Results folder: C:\Users\gadha\sakura_analytics\outputs\baseline


In [12]:
feb_df = load_features("feb1", source="csv")
mar_df = load_features("mar1", source="csv")

train = feb_df.loc[feb_df["bloom_year"].between(1964, 2014)].copy()
test_feb = feb_df.loc[feb_df["bloom_year"].between(2015, 2025)].copy()
test_mar = mar_df.loc[mar_df["bloom_year"].between(2015, 2025)].copy()

KEYS = ["station_id", "bloom_year"]
assert not test_feb.duplicated(KEYS).any() and not test_mar.duplicated(KEYS).any()
assert set(map(tuple, test_feb[KEYS].to_numpy())) == set(map(tuple, test_mar[KEYS].to_numpy()))
assert len(test_feb) == len(test_mar) == 528
assert train["bloom_year"].max() == 2014
print("Development rows:", len(train), "| Test rows per date:", len(test_feb))

Development rows: 3837 | Test rows per date: 528


In [13]:
#Freeze the decisions
CONFIG = {
    "train_years": [1964, 2014], "test_years": [2015, 2025],
    "station_lookback": 30, "minimum_station_history": 10,
    "station_fallback": "regional_then_national",
    "temperature_lookback": 30,
    "accumulation_start": "02-01", "accumulation_end": "06-30",
    "threshold_calibration": "training_median_at_observed_full_bloom",
    "threshold_variants": ["national", "regional"],
    "issue_dates": ["feb1", "mar1"],
    "metric_aggregation": "single_held_out_test_sample",
    "selected_before_test": True
}
config_path = OUTPUT / "baseline_frozen_config.json"
config_path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Frozen configuration saved:", config_path)

Frozen configuration saved: C:\Users\gadha\sakura_analytics\outputs\baseline\baseline_frozen_config.json


In [14]:
#Frozen model implementations
def climatology_30yr(train_df, predict_df):
    predictions = []
    for row in predict_df.itertuples(index=False):
        history = train_df[train_df["bloom_year"].between(row.bloom_year - 30, row.bloom_year - 1)]
        station_history = history[history["station_id"] == row.station_id]
        if len(station_history) >= 10:
            prediction = station_history[TARGET].mean()
        else:
            region_history = history[history["region"] == row.region]
            if len(region_history):
                prediction = region_history[TARGET].mean()
            elif len(history):
                prediction = history[TARGET].mean()
            else:
                prediction = np.nan
        predictions.append(prediction)
    return np.asarray(predictions, dtype=float)


def calculate_full_bloom_accumulation(train_df, weather_df):
    daily = weather_df.sort_values(["station_id", "date"]).copy()
    daily["valid_tmax"] = daily["maximum_temperature_c"].notna().astype(int)
    groups = daily.groupby(["station_id", "year"])
    daily["cumulative_tmax"] = groups["maximum_temperature_c"].cumsum()
    daily["cumulative_valid_days"] = groups["valid_tmax"].cumsum()
    targets = train_df[["station_id", "bloom_year", "full_bloom_date", TARGET]].copy()
    targets["station_id"] = targets["station_id"].astype(str)
    targets["full_bloom_date"] = pd.to_datetime(targets["full_bloom_date"])
    merged = targets.merge(
        daily[["station_id", "date", "cumulative_tmax", "cumulative_valid_days"]],
        left_on=["station_id", "full_bloom_date"], right_on=["station_id", "date"],
        how="left", validate="one_to_one"
    )
    start_dates = pd.to_datetime(merged["bloom_year"].astype(str) + "-02-01")
    merged["expected_days"] = (merged["full_bloom_date"] - start_dates).dt.days + 1
    complete = merged.loc[(merged["expected_days"] > 0) &
                          (merged["cumulative_valid_days"] == merged["expected_days"])].copy()
    return complete[["station_id", "bloom_year", TARGET, "cumulative_tmax"]].rename(
        columns={"cumulative_tmax": "accumulated_tmax"})


def prepare_daily_climatology(weather_df, train_end_year, lookback=30):
    history = weather_df.loc[weather_df["year"].between(
        train_end_year - lookback + 1, train_end_year)].dropna(subset=["maximum_temperature_c"])
    station = history.groupby(["station_id", "month_day"])["maximum_temperature_c"].mean()
    national = history.groupby("month_day")["maximum_temperature_c"].mean()
    return station, national


def predict_temperature_rule(station_id, year, issue_date, threshold,
                             station_clim, national_clim, observed_lookup):
    station_id = str(station_id)
    cumulative = 0.0
    estimated_days = fallback_days = missing_observed_days = 0
    for date in pd.date_range(pd.Timestamp(year, 2, 1), pd.Timestamp(year, 6, 30)):
        month_day = date.strftime("%m-%d")
        fallback_day = "02-28" if month_day == "02-29" else month_day
        estimated = station_clim.get((station_id, month_day), np.nan)
        if pd.isna(estimated):
            estimated = station_clim.get((station_id, fallback_day), np.nan)
        if pd.isna(estimated):
            estimated = national_clim.get(month_day, np.nan)
            fallback_days += 1
        if pd.isna(estimated):
            estimated = national_clim.get(fallback_day, np.nan)
        if date < issue_date:
            actual = observed_lookup.get((station_id, date), np.nan)
            if pd.notna(actual):
                temperature = actual
            else:
                temperature = estimated
                missing_observed_days += 1
                estimated_days += 1
        else:
            temperature = estimated
            estimated_days += 1
        if pd.isna(temperature):
            return {"predicted_doy": np.nan, "estimated_days": estimated_days,
                    "fallback_days": fallback_days, "missing_observed_days": missing_observed_days,
                    "status": "missing_temperature"}
        cumulative += float(temperature)
        if cumulative >= threshold:
            return {"predicted_doy": date.dayofyear, "estimated_days": estimated_days,
                    "fallback_days": fallback_days, "missing_observed_days": missing_observed_days,
                    "status": "crossed"}
    return {"predicted_doy": np.nan, "estimated_days": estimated_days,
            "fallback_days": fallback_days, "missing_observed_days": missing_observed_days,
            "status": "threshold_not_reached"}

print("Frozen baseline functions defined.")

Frozen baseline functions defined.


In [15]:
#Load daily weather and audit permitted observed February inputs
WEATHER_PATH = (ROOT / "data" / "processed" / "02_cleaned_source_data" /
                "weather" / "02_cleaned_weather_daily_somei_1952_2025.csv.gz")
weather = pd.read_csv(
    WEATHER_PATH,
    usecols=["station_id", "date", "maximum_temperature_c", "temp_imputed"],
    dtype={"station_id": str}, parse_dates=["date"]
)
weather["year"] = weather["date"].dt.year
weather = weather.loc[weather["date"].dt.month.between(2, 6)].copy()
weather["month_day"] = weather["date"].dt.strftime("%m-%d")
assert not weather.duplicated(["station_id", "date"]).any(), "Duplicate station/date in weather"
print("Daily weather rows Feb–Jun:", len(weather))

Daily weather rows Feb–Jun: 911758


In [22]:
#fit the frozen parameters on development data only
calibration = calculate_full_bloom_accumulation(train, weather)
assert not calibration.empty
calibration = calibration.merge(
    train[["station_id", "bloom_year", "region"]],
    on=["station_id", "bloom_year"], how="left", validate="one_to_one")
assert calibration["region"].notna().all()
national_threshold = float(calibration["accumulated_tmax"].median())
regional_thresholds = calibration.groupby("region")["accumulated_tmax"].median()
station_clim, national_clim = prepare_daily_climatology(weather, train_end_year=2014, lookback=30)
assert regional_thresholds.notna().all()

params = {
    "fit_through_year": 2014,
    "national_threshold": national_threshold,
    "regional_thresholds": {str(k): float(v) for k, v in regional_thresholds.items()},
    "calibration_observations": int(len(calibration))
}
(OUTPUT / "baseline_frozen_parameters.json").write_text(json.dumps(params, indent=2), encoding="utf-8")
station_clim.rename("estimated_tmax").to_csv(OUTPUT / "frozen_station_temperature_climatology.csv")
national_clim.rename("estimated_tmax").to_csv(OUTPUT / "frozen_national_temperature_climatology.csv")
print("Frozen national threshold:", round(national_threshold, 2))
display(regional_thresholds.round(2))
print("Parameters saved before test scoring.")

Frozen national threshold: 746.1


region
Chubu       717.40
Chugoku     747.20
Hokkaido    573.80
Kansai      762.35
Kanto       786.05
Kyushu      845.00
Shikoku     799.20
Tohoku      664.70
Name: accumulated_tmax, dtype: float64

Parameters saved before test scoring.


In [16]:
# PRE-TEST INPUT AUDIT — no target values used here.
# The existing March-1 forecast reads observed February daily Tmax.
# Flag imputed observations: their upstream interpolation may have used post-cutoff weather.
test_keys = test_mar[["station_id", "bloom_year"]].copy()
test_keys["station_id"] = test_keys["station_id"].astype(str)
observed_feb = weather.loc[
    weather["date"].dt.month.eq(2),
    ["station_id", "year", "date", "maximum_temperature_c", "temp_imputed"]
].merge(test_keys, left_on=["station_id", "year"],
        right_on=["station_id", "bloom_year"], how="inner", validate="many_to_one")
flagged = observed_feb.loc[observed_feb["temp_imputed"].fillna(0).ne(0)]
missing = observed_feb["maximum_temperature_c"].isna().sum()
print("Test February observed rows:", len(observed_feb))
print("Flagged imputed February temperatures:", len(flagged))
print("Missing February Tmax in existing rows:", missing)
if len(flagged):
    display(flagged.head(10))
# Stop before test prediction if weather provenance is unresolved.
assert flagged.empty, (
    "Some observed test-February temperatures are flagged as imputed. "
    "Audit whether their imputation uses post-issue data before proceeding."
)
print("Input flag check passed. Note: unflagged ETL is not independently proven leakage-free.")

Test February observed rows: 14928
Flagged imputed February temperatures: 0
Missing February Tmax in existing rows: 0
Input flag check passed. Note: unflagged ETL is not independently proven leakage-free.


In [18]:
#Generate all held-out predictions
observed_lookup = {
    (r.station_id, r.date): r.maximum_temperature_c
    for r in weather[["station_id", "date", "maximum_temperature_c"]].itertuples(index=False)
}

rows = []
for issue, test_df in [("feb1", test_feb), ("mar1", test_mar)]:
    clim_pred = climatology_30yr(train, test_df)
    for row, cpred in zip(test_df.itertuples(index=False), clim_pred):
        year = int(row.bloom_year)
        issue_date = pd.Timestamp(year, 2 if issue == "feb1" else 3, 1)
        rows.append({"model": "30yr_climatology", "forecast_date": issue,
                     "station_id": str(row.station_id), "year": year,
                     "predicted_doy": float(cpred), "status": "predicted"})
        for variant in ("national", "regional"):
            threshold = (national_threshold if variant == "national"
                         else float(regional_thresholds.get(row.region, national_threshold)))
            result = predict_temperature_rule(
                row.station_id, year, issue_date, threshold,
                station_clim, national_clim, observed_lookup
            )
            rows.append({"model": f"{variant}_temperature_rule", "forecast_date": issue,
                         "station_id": str(row.station_id), "year": year,
                         "threshold": threshold, **result})

predictions = pd.DataFrame(rows)
print("Predictions per model/date:")
display(predictions.groupby(["model", "forecast_date", "status"]).size().rename("n"))
# Preserve all predictions before evaluation.
predictions.to_csv(OUTPUT / "baseline_test_predictions_unscored.csv", index=False)

Predictions per model/date:


model                      forecast_date  status   
30yr_climatology           feb1           predicted    528
                           mar1           predicted    528
national_temperature_rule  feb1           crossed      528
                           mar1           crossed      528
regional_temperature_rule  feb1           crossed      528
                           mar1           crossed      528
Name: n, dtype: int64

In [19]:
#Pre-scoring validity gate
assert not predictions.duplicated(["model", "forecast_date", "station_id", "year"]).any()
assert predictions["predicted_doy"].notna().all(), "A baseline has missing test predictions"
assert predictions.groupby(["model", "forecast_date"]).size().eq(528).all()
issue_dates = pd.to_datetime(predictions["year"].astype(str) + np.where(
    predictions["forecast_date"].eq("feb1"), "-02-01", "-03-01"))
invalid_issue = predictions["predicted_doy"] < issue_dates.dt.dayofyear
print("Predictions before issue date:", int(invalid_issue.sum()))
assert not invalid_issue.any(), "Pre-issue threshold crossing: review without retuning on test outcomes"
print("All test predictions passed structural checks. Ready for one-time scoring.")

Predictions before issue date: 0
All test predictions passed structural checks. Ready for one-time scoring.


In [20]:
#One-time final evaluation with the team metrics
target_lookup = feb_df.loc[feb_df["bloom_year"].between(2015, 2025),
                           ["station_id", "bloom_year", TARGET]].copy()
target_lookup["station_id"] = target_lookup["station_id"].astype(str)
target_lookup = target_lookup.rename(columns={"bloom_year": "year", TARGET: "actual_doy"})
scored = predictions.merge(target_lookup, on=["station_id", "year"],
                           how="left", validate="many_to_one")
assert scored["actual_doy"].notna().all()
metrics = []
for (model, issue), group in scored.groupby(["model", "forecast_date"]):
    metrics.append({"model": model, "forecast_date": issue,
                    **evaluate(group["actual_doy"], group["predicted_doy"])})
metrics = pd.DataFrame(metrics).sort_values(["forecast_date", "MAE"])
display_metrics = metrics.copy()
display_metrics["hit_rate_3d"] *= 100
print("FINAL HELD-OUT TEST (2015–2025) — hit rate displayed as %")
display(display_metrics.round(3))

FINAL HELD-OUT TEST (2015–2025) — hit rate displayed as %


,model,forecast_date,n,MAE,RMSE,R2,hit_rate_3d
0,30yr_climatology,feb1,528,5.040,6.194,0.500,35.227
4,regional_temperature_rule,feb1,528,5.915,7.261,0.312,34.280
2,national_temperature_rule,feb1,528,7.339,9.385,-0.149,32.008
1,30yr_climatology,mar1,528,5.040,6.194,0.500,35.227
5,regional_temperature_rule,mar1,528,5.555,6.859,0.387,37.500
3,national_temperature_rule,mar1,528,7.157,9.035,-0.064,29.735


In [21]:
scored.to_csv(OUTPUT / "baseline_test_predictions.csv", index=False)
metrics.to_csv(OUTPUT / "baseline_test_metrics.csv", index=False)
print("Saved baseline_test_predictions.csv and baseline_test_metrics.csv")
print("Reminder: these test results are final. Do not change configurations based on test performance.")

Saved baseline_test_predictions.csv and baseline_test_metrics.csv
Reminder: these test results are final. Do not change configurations based on test performance.


In [23]:

# ============================================================
# FINAL BASELINE TESTING — SUMMARY AND KEY FINDINGS
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Load actual final test results
results = pd.read_csv(
    OUTPUT / "baseline_test_metrics.csv"
)

# Display results for inspection
display(results.round(3))

# Identify model and metric columns
print("Available columns:", results.columns.tolist())


,model,forecast_date,n,MAE,RMSE,R2,hit_rate_3d
0,30yr_climatology,feb1,528,5.040,6.194,0.500,0.352
1,regional_temperature_rule,feb1,528,5.915,7.261,0.312,0.343
2,national_temperature_rule,feb1,528,7.339,9.385,-0.149,0.320
3,30yr_climatology,mar1,528,5.040,6.194,0.500,0.352
4,regional_temperature_rule,mar1,528,5.555,6.859,0.387,0.375
5,national_temperature_rule,mar1,528,7.157,9.035,-0.064,0.297


Available columns: ['model', 'forecast_date', 'n', 'MAE', 'RMSE', 'R2', 'hit_rate_3d']


In [26]:

# ============================================================
# AUTOMATED FINAL TEST REPORT
# ============================================================

results = pd.read_csv(
    OUTPUT / "baseline_test_metrics.csv"
)

# Find model label column
model_col = next(
    (c for c in ["model", "model_name", "baseline"]
     if c in results.columns),
    None
)

if model_col is None:
    raise ValueError(
        "Model label column not found. "
        f"Available columns: {results.columns.tolist()}"
    )

# Identify best models
best_mae = results.loc[results["MAE"].idxmin()]
best_rmse = results.loc[results["RMSE"].idxmin()]
best_r2 = results.loc[results["R2"].idxmax()]
best_hit = results.loc[results["hit_rate_3d"].idxmax()]

# Format comparison table
display_table = results.copy()

# Shared framework stores hit rate as a proportion
display_table["hit_rate_3d"] *= 100

summary_table = display_table.to_markdown(index=False)

report = f"""
# Final Baseline Testing — Summary and Key Findings

## 1. Evaluation Overview

The selected Sakura Yosoku baseline configurations were
evaluated on the held-out **2015–2025 test period**.

All model configurations were selected using the
five expanding-window validation folds (2005–2014).

The final test evaluation used:

- **Target:** Full-bloom day of year (DOY)
- **Species:** Somei-yoshino
- **Training period:** 1964–2014
- **Test period:** 2015–2025
- **Evaluation metrics:** MAE, RMSE, R² and ±3-day hit rate

The test dataset was reserved during model development.

---

## 2. Final Test Performance

{summary_table}

*MAE and RMSE are measured in days.
The ±3-day hit rate is displayed as a percentage.*

---

## 3. Key Findings

### Finding 1 — Lowest Average Prediction Error

**{best_mae[model_col]}** achieved the lowest
test MAE of **{best_mae['MAE']:.3f} days**.

This model produced the smallest average absolute
difference between predicted and actual full-bloom dates.

### Finding 2 — Lowest Large-Error Penalty

**{best_rmse[model_col]}** achieved the lowest
RMSE of **{best_rmse['RMSE']:.3f} days**.

RMSE penalizes larger prediction errors more heavily
than MAE.

### Finding 3 — Highest Explained Variation

**{best_r2[model_col]}** achieved the highest
R² of **{best_r2['R2']:.3f}**.

This indicates the strongest test-set performance
relative to the mean-target prediction reference.

### Finding 4 — Highest ±3-Day Hit Rate

**{best_hit[model_col]}** achieved the highest
±3-day hit rate of
**{best_hit['hit_rate_3d'] * 100:.2f}%**.

This represents the proportion of predictions
within three days of the observed full-bloom date.

---

## 4. Interpretation

The final test results provide an independent
assessment of the selected baseline configurations
on observations from 2015–2025.

The lowest-MAE model is the strongest baseline
for minimizing average prediction error.

The highest-hit-rate model is the strongest baseline
for satisfying the project's ±3-day forecasting
tolerance.

These objectives may favour different models.

---

## 5. Limitations

- Station Climatology relies on historical bloom
  timing and does not incorporate current-season weather.
- Temperature-rule forecasts rely on historical
  estimates of future daily temperatures.
- Regional temperature thresholds may not capture
  station-specific microclimatic differences.
- Forecast accuracy may vary across regions and years.
- Results should be interpreted alongside forecast
  coverage and any unresolved preprocessing audits.
- Test results must not be used for additional
  hyperparameter tuning or model selection.

---

## 6. Conclusion

The held-out test evaluation establishes the final
performance of the Sakura Yosoku baseline models.

**Best test MAE:** {best_mae[model_col]}
({best_mae['MAE']:.3f} days).

**Best ±3-day hit rate:** {best_hit[model_col]}
({best_hit['hit_rate_3d'] * 100:.2f}%).

These results provide the reference benchmarks
for comparing the team's selected champion models
on the same 2015–2025 test observations.
"""

display(Markdown(report))



# Final Baseline Testing — Summary and Key Findings

## 1. Evaluation Overview

The selected Sakura Yosoku baseline configurations were
evaluated on the held-out **2015–2025 test period**.

All model configurations were selected using the
five expanding-window validation folds (2005–2014).

The final test evaluation used:

- **Target:** Full-bloom day of year (DOY)
- **Species:** Somei-yoshino
- **Training period:** 1964–2014
- **Test period:** 2015–2025
- **Evaluation metrics:** MAE, RMSE, R² and ±3-day hit rate

The test dataset was reserved during model development.

---

## 2. Final Test Performance

| model                     | forecast_date   |   n |     MAE |    RMSE |         R2 |   hit_rate_3d |
|:--------------------------|:----------------|----:|--------:|--------:|-----------:|--------------:|
| 30yr_climatology          | feb1            | 528 | 5.03959 | 6.1941  |  0.49967   |       35.2273 |
| regional_temperature_rule | feb1            | 528 | 5.91477 | 7.26149 |  0.312375  |       34.2803 |
| national_temperature_rule | feb1            | 528 | 7.33902 | 9.38517 | -0.148642  |       32.0076 |
| 30yr_climatology          | mar1            | 528 | 5.03959 | 6.1941  |  0.49967   |       35.2273 |
| regional_temperature_rule | mar1            | 528 | 5.55492 | 6.85855 |  0.386569  |       37.5    |
| national_temperature_rule | mar1            | 528 | 7.1572  | 9.03476 | -0.0644708 |       29.7348 |

*MAE and RMSE are measured in days.
The ±3-day hit rate is displayed as a percentage.*

---

## 3. Key Findings

### Finding 1 — Lowest Average Prediction Error

**30yr_climatology** achieved the lowest
test MAE of **5.040 days**.

This model produced the smallest average absolute
difference between predicted and actual full-bloom dates.

### Finding 2 — Lowest Large-Error Penalty

**30yr_climatology** achieved the lowest
RMSE of **6.194 days**.

RMSE penalizes larger prediction errors more heavily
than MAE.

### Finding 3 — Highest Explained Variation

**30yr_climatology** achieved the highest
R² of **0.500**.

This indicates the strongest test-set performance
relative to the mean-target prediction reference.

### Finding 4 — Highest ±3-Day Hit Rate

**regional_temperature_rule** achieved the highest
±3-day hit rate of
**37.50%**.

This represents the proportion of predictions
within three days of the observed full-bloom date.

---

## 4. Interpretation

The final test results provide an independent
assessment of the selected baseline configurations
on observations from 2015–2025.

The lowest-MAE model is the strongest baseline
for minimizing average prediction error.

The highest-hit-rate model is the strongest baseline
for satisfying the project's ±3-day forecasting
tolerance.

These objectives may favour different models.

---

## 5. Limitations

- Station Climatology relies on historical bloom
  timing and does not incorporate current-season weather.
- Temperature-rule forecasts rely on historical
  estimates of future daily temperatures.
- Regional temperature thresholds may not capture
  station-specific microclimatic differences.
- Forecast accuracy may vary across regions and years.
- Results should be interpreted alongside forecast
  coverage and any unresolved preprocessing audits.
- Test results must not be used for additional
  hyperparameter tuning or model selection.

---

## 6. Conclusion

The held-out test evaluation establishes the final
performance of the Sakura Yosoku baseline models.

**Best test MAE:** 30yr_climatology
(5.040 days).

**Best ±3-day hit rate:** regional_temperature_rule
(37.50%).

These results provide the reference benchmarks
for comparing the team's selected champion models
on the same 2015–2025 test observations.
